# Exploratory Data Analysis (EDA)

**Goal:** explore the curated training/test tables and build baseline + improve regression models for predicting `fare_amount`.

**What you will find here:**
- Data loading from Unity Catalog (`final_project.gold.train`, `final_project.gold.test`)
- Basic quality checks and EDA (distributions, correlations, feature behavior)
- Feature preparation for modeling
- Model training (baselines + tree-based models where applicable) and evaluation


%md
## 1. Imports

In [0]:
pip install folium xgboost lightgbm

In [0]:
# Consolidated imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression as SparkLinearRegression
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import folium
from pyspark.sql.types import NumericType

## 2. Data Loading

In [0]:
# Load data from the Lakehouse/Unity Catalog tables.

train_df = spark.table("final_project.gold.train")
test_df  = spark.table("final_project.gold.test")

## 3. Data Overview & Basic Quality Checks

Let's check basic dataset dimensions and schemas to validate that data loading worked as expected.

In [0]:
# Quick shape and schema checks.
train_rows, train_cols = train_df.count(), len(train_df.columns)
test_rows,  test_cols  = test_df.count(),  len(test_df.columns)

print(f"Train shape: {train_rows} rows, {train_cols} columns")
print(f"Test shape:  {test_rows} rows, {test_cols} columns")


In [0]:
print("\nTrain schema:")
train_df.printSchema()

In [0]:
print("\nTest schema:")
test_df.printSchema()

In [0]:
# Preview a few rows
display(train_df.limit(5))

In [0]:
# Descriptive stats for numeric columns
display(train_df.describe())


### 3.1 Missing values report

In [0]:
# Missing values per column

def null_report(df):
    exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    out = df.select(exprs)
    # Convert a single-row Spark DF to a long Pandas table for readability
    pdf = out.toPandas().T.reset_index()
    pdf.columns = ["column", "null_count"]
    return pdf.sort_values("null_count", ascending=False)

train_nulls = null_report(train_df)
test_nulls  = null_report(test_df)
print("Train nulls:")
display(train_nulls[train_nulls["null_count"] > 0])
print("Test nulls:")
display(test_nulls[test_nulls["null_count"] > 0])


## 4. One-variable analysis

In [0]:
SAMPLE_FRAC = 0.05
SEED = 42

train_pd = train_df.sample(withReplacement=False, fraction=SAMPLE_FRAC, seed=SEED).toPandas()
test_pd  = test_df.sample(withReplacement=False, fraction=SAMPLE_FRAC, seed=SEED).toPandas()

print(train_pd.shape, test_pd.shape)

Let's plot feature/target distributions to compare groups and understand skewness and tails.

In [0]:
if "fare_amount" in train_pd.columns:
    fare = train_pd["fare_amount"].dropna()

    plt.figure(figsize=(10, 4))
    plt.hist(fare, bins=80, edgecolor="black", linewidth=0.4)
    plt.title("Fare Amount Distribution (Train)")
    plt.xlabel("Fare Amount")
    plt.ylabel("Trip Count")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [0]:
if "fare_amount" in train_pd.columns:
    fare = train_pd["fare_amount"].dropna()

    q01, q99 = fare.quantile([0.01, 0.99])
    fare_clip = fare.clip(lower=q01, upper=q99)

    plt.figure(figsize=(10, 4))
    plt.hist(fare_clip, bins=80, edgecolor="black", linewidth=0.4)
    plt.title("Fare Amount Distribution (Train, 1st–99th Percentile)")
    plt.xlabel("Fare Amount")
    plt.ylabel("Trip Count")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [0]:
# Weather feature distributions

for col in ["Mean_TemperatureF", "Mean_Wind_SpeedMPH"]:
    if col in train_pd.columns:
        x = train_pd[col].dropna()

        plt.figure(figsize=(10, 4))
        plt.hist(x, bins=60, edgecolor="black", linewidth=0.4)
        plt.title(f"{col} Distribution (Train Sample)")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()


In [0]:
distance_cols = [
    c for c in ["hs_dist", "jfk_drop_distance", "lga_drop_distance", "ewr_drop_distance"]
    if c in train_pd.columns
]

for col in distance_cols:
    x = train_pd[col].dropna()

    plt.figure(figsize=(10, 4))
    plt.hist(x, bins=80, edgecolor="black", linewidth=0.4)
    plt.title(f"{col} Distribution (Train Sample)")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [0]:
# Distance-related features (often heavy-tailed).
distance_cols = [
    c for c in ["hs_dist", "jfk_drop_distance", "lga_drop_distance", "ewr_drop_distance"]
    if c in train_pd.columns
]

for col in distance_cols:
    x = train_pd[col].dropna()

    plt.figure(figsize=(10, 4))
    plt.hist(x, bins=80, log=True, edgecolor="black", linewidth=0.4)
    plt.title(f"{col} Distribution (Train Sample, Log Scale)")
    plt.xlabel(col)
    plt.ylabel("Count (log)")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


## 5. Multivariate & Correlation Analysis

In [0]:
TARGET = "fare_amount"

# 1. Explicitly define columns where Pearson correlation is NOT meaningful
EXCLUDE_COLS = {
    TARGET,
    "pickup_region",
    "dropoff_region",
    "pickup_datetime",
    "key",
    "month",
    "day_of_week",
    "day",
}

# exclude one-hot month/day columns
EXCLUDE_PREFIXES = (
    "jan", "feb", "mar", "apr", "may", "june",
    "july", "aug", "sep", "oct", "nov", "dec",
    "mon", "tue", "wed", "thu", "fri", "sat", "sun"
)

# 2. Collect numeric columns
numeric_cols = [
    f.name
    for f in train_df.schema.fields
    if isinstance(f.dataType, NumericType)
]

# 3. Keep only columns suitable for Pearson correlation
corr_features = [
    c for c in numeric_cols
    if c not in EXCLUDE_COLS
    and not c.startswith(EXCLUDE_PREFIXES)
]

# 4. Compute Pearson correlation with target
corr_rows = []
for c in corr_features:
    val = train_df.select(F.corr(c, TARGET).alias("corr")).collect()[0]["corr"]
    corr_rows.append((c, float(val) if val is not None else None))

corr_df = (
    spark.createDataFrame(corr_rows, ["feature", "pearson_corr"])
    .orderBy(F.abs(F.col("pearson_corr")).desc())
)

display(corr_df)


In [0]:
cols_for_matrix = [TARGET] + corr_features

sample_pd = (
    train_df.select(*cols_for_matrix)
    .dropna(subset=[TARGET])  # keep target present
    .sample(withReplacement=False, fraction=0.05, seed=42)  # adjust fraction if needed
    .limit(20000)
    .toPandas()
)

corr_mat = sample_pd.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_mat.values, aspect="auto", interpolation="nearest")
fig.colorbar(im, ax=ax, label="Pearson correlation")

ax.set_xticks(range(len(corr_mat.columns)))
ax.set_yticks(range(len(corr_mat.columns)))
ax.set_xticklabels(corr_mat.columns, rotation=90)
ax.set_yticklabels(corr_mat.columns)

ax.grid(False)
ax.minorticks_off()

ax.set_title("Correlation matrix (eligible numeric features, sampled)")
fig.tight_layout()
plt.show()

### Geography: pickup/dropoff maps by region

In [0]:
geo_cols = {"pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude", "pickup_region", "dropoff_region", "fare_amount", "hs_dist"}
if geo_cols.issubset(set(train_df.columns)):

    cols = ["pickup_latitude","pickup_longitude","dropoff_latitude","dropoff_longitude","pickup_region","dropoff_region","fare_amount","hs_dist"]

    base = (
        train_df.select(*cols)
        .where(
            F.col("pickup_latitude").isNotNull() &
            F.col("pickup_longitude").isNotNull() &
            F.col("dropoff_latitude").isNotNull() &
            F.col("dropoff_longitude").isNotNull() &
            F.col("pickup_region").isNotNull() &
            F.col("dropoff_region").isNotNull()
        )
    )

    # Sample up to N points per region to avoid overplotting.
    N_PER_REGION = 250
    w_pick = Window.partitionBy("pickup_region").orderBy(F.rand(SEED))
    w_drop = Window.partitionBy("dropoff_region").orderBy(F.rand(SEED))

    pick = base.withColumn("rn", F.row_number().over(w_pick)).where(F.col("rn") <= N_PER_REGION).drop("rn")
    drop = base.withColumn("rn", F.row_number().over(w_drop)).where(F.col("rn") <= N_PER_REGION).drop("rn")

    pdf_pick = pick.toPandas()
    pdf_drop = drop.toPandas()

    center_lat = float(pdf_pick["pickup_latitude"].mean())
    center_lon = float(pdf_pick["pickup_longitude"].mean())

    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
        "#3182bd", "#31a354", "#756bb1", "#636363", "#e6550d"
    ]

    def region_color(region_id: int) -> str:
        return palette[int(region_id) % len(palette)]

    m_pickup = folium.Map([center_lat, center_lon], zoom_start=11, tiles="CartoDB positron")
    for r in pdf_pick.itertuples():
        folium.CircleMarker(
            [r.pickup_latitude, r.pickup_longitude],
            radius=3,
            color=region_color(r.pickup_region),
            fill=True,
            fill_opacity=0.7,
            popup=f"pickup_region={int(r.pickup_region)}, fare={r.fare_amount:.2f}, hs_dist={r.hs_dist:.2f}"
        ).add_to(m_pickup)

    m_dropoff = folium.Map([center_lat, center_lon], zoom_start=11, tiles="CartoDB positron")
    for r in pdf_drop.itertuples():
        folium.CircleMarker(
            [r.dropoff_latitude, r.dropoff_longitude],
            radius=3,
            color=region_color(r.dropoff_region),
            fill=True,
            fill_opacity=0.7,
            popup=f"dropoff_region={int(r.dropoff_region)}, fare={r.fare_amount:.2f}, hs_dist={r.hs_dist:.2f}"
        ).add_to(m_dropoff)
else:
    print("Geography columns not found; skipping Folium maps.")


In [0]:
display(m_pickup)

In [0]:
display(m_dropoff)

## 6. Key Insights / Deep-Dive Visualizations

### 6.1 Trend analysis over time

In [0]:
if "pickup_datetime" in train_df.columns:
    df_t = (
        train_df
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
        .withColumn("pickup_month", F.date_trunc("month", F.col("pickup_datetime")).cast("date"))
    )
elif {"year", "month", "day"}.issubset(set(train_df.columns)):
    df_t = (
        train_df
        .withColumn("pickup_date", F.to_date(F.concat_ws("-", F.col("year"), F.col("month"), F.col("day"))))
        .withColumn("pickup_month", F.to_date(F.concat_ws("-", F.col("year"), F.col("month"), F.lit(1))))
    )
else:
    df_t = None

required = {"pickup_date", "fare_amount", "hs_dist"}
if df_t is not None and required.issubset(set(df_t.columns)):
    daily = (
        df_t.groupBy("pickup_date")
        .agg(
            F.count("*").alias("daily_trips"),
            F.avg("fare_amount").alias("daily_mean_fare"),
            F.expr("percentile_approx(fare_amount, 0.5)").alias("daily_median_fare"),
            F.avg("hs_dist").alias("daily_mean_distance"),
            F.expr("percentile_approx(hs_dist, 0.5)").alias("daily_median_distance"),
        )
        .orderBy("pickup_date")
    )

    daily_pd = daily.toPandas()

    plt.figure(figsize=(12, 4))
    plt.plot(daily_pd["pickup_date"], daily_pd["daily_trips"], linewidth=1.6)
    plt.title("Daily Number of Trips Over Time (Train)")
    plt.xlabel("Pickup Date")
    plt.ylabel("Trips")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    monthly = (
        df_t.groupBy("pickup_month")
        .agg(
            F.count("*").alias("monthly_trips"),
            F.avg("fare_amount").alias("monthly_mean_fare"),
            F.avg("hs_dist").alias("monthly_mean_distance"),
        )
        .orderBy("pickup_month")
    )
else:
    print("Time trend columns not found; skipping trend analysis.")

In [0]:
monthly_pd = monthly.toPandas().sort_values("pickup_month")

plt.figure(figsize=(12, 4))
plt.plot(monthly_pd["pickup_month"], monthly_pd["monthly_trips"], linewidth=1.6)
plt.title("Monthly Number of Trips Over Time (Train)")
plt.xlabel("Pickup Month")
plt.ylabel("Trips")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(monthly_pd["pickup_month"], monthly_pd["monthly_mean_fare"], linewidth=1.6)
plt.title("Monthly Mean Fare Over Time (Train)")
plt.xlabel("Pickup Month")
plt.ylabel("Mean Fare Amount")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(monthly_pd["pickup_month"], monthly_pd["monthly_mean_distance"], linewidth=1.6)
plt.title("Monthly Mean Distance Over Time (Train)")
plt.xlabel("Pickup Month")
plt.ylabel("Mean Distance (hs_dist)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 Hourly demand and average fare


In [0]:
if "hour" in train_df.columns:
    hourly_stats = (
        train_df.groupBy("hour")
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("fare_amount").alias("avg_fare_amount"),
        )
        .orderBy("hour")
    )

    try:
        display(hourly_stats)
    except NameError:
        hourly_stats.show(24, truncate=False)

    hourly_pd = hourly_stats.toPandas()

    plt.figure(figsize=(10, 4))
    plt.plot(hourly_pd["hour"], hourly_pd["trip_count"], linewidth=1.6, marker="o", markersize=3)
    plt.title("Trips by Hour (Train)")
    plt.xlabel("Hour of Day")
    plt.ylabel("Number of Trips")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(hourly_pd["hour"], hourly_pd["avg_fare_amount"], linewidth=1.6, marker="o", markersize=3)
    plt.title("Average Fare by Hour (Train)")
    plt.xlabel("Hour of Day")
    plt.ylabel("Average Fare Amount")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Column 'hour' not found in train_df; skipping hourly analysis.")

### 6.2 Weather impact (cold_weather flag)

In [0]:
if {"cold_weather", "fare_amount"}.issubset(set(train_df.columns)):
    cold_sample = (
        train_df.select("cold_weather", "fare_amount")
        .sample(False, SAMPLE_FRAC, SEED)
        .toPandas()
    )

    fare_c0 = cold_sample.loc[cold_sample["cold_weather"] == 0, "fare_amount"].dropna()
    fare_c1 = cold_sample.loc[cold_sample["cold_weather"] == 1, "fare_amount"].dropna()

    plt.figure(figsize=(10, 4))
    plt.hist(fare_c0, bins=80, density=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="cold_weather = 0")
    plt.hist(fare_c1, bins=80, density=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="cold_weather = 1")
    plt.title("Fare Distribution: Cold vs Not Cold (Train Sample)")
    plt.xlabel("Fare Amount")
    plt.ylabel("Density")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.hist(fare_c0, bins=80, density=True, log=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="cold_weather = 0")
    plt.hist(fare_c1, bins=80, density=True, log=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="cold_weather = 1")
    plt.title("Fare Distribution: Cold vs Not Cold (Log Scale, Train Sample)")
    plt.xlabel("Fare Amount")
    plt.ylabel("Density (log)")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    bad_weather_stats = (
        train_df.groupBy("cold_weather")
        .agg(
            F.count("*").alias("n"),
            F.avg("fare_amount").alias("mean_fare"),
            F.avg("hs_dist").alias("mean_dist"),
        )
        .orderBy("cold_weather")
    )

    try:
        display(bad_weather_stats)
    except NameError:
        bad_weather_stats.show(10, truncate=False)
else:
    print("Weather flag columns not found; skipping weather impact section.")

### 6.3 Weather impact (example: bad_meather flag)

In [0]:
if {"bad_weather", "fare_amount"}.issubset(set(train_df.columns)):
    bad_sample = (
        train_df.select("bad_weather", "fare_amount")
        .sample(False, SAMPLE_FRAC, SEED)
        .toPandas()
    )

    fare_b0 = bad_sample.loc[bad_sample["bad_weather"] == 0, "fare_amount"].dropna()
    fare_b1 = bad_sample.loc[bad_sample["bad_weather"] == 1, "fare_amount"].dropna()

    plt.figure(figsize=(10, 4))
    plt.hist(fare_b0, bins=80, density=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="bad_weather = 0")
    plt.hist(fare_b1, bins=80, density=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="bad_weather = 1")
    plt.title("Fare Distribution: Bad Weather vs Not Bad (Train Sample)")
    plt.xlabel("Fare Amount")
    plt.ylabel("Density")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.hist(fare_b0, bins=80, density=True, log=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="bad_weather = 0")
    plt.hist(fare_b1, bins=80, density=True, log=True, alpha=0.6, edgecolor="black", linewidth=0.3, label="bad_weather = 1")
    plt.title("Fare Distribution: Bad Weather vs Not Bad (Log Scale, Train Sample)")
    plt.xlabel("Fare Amount")
    plt.ylabel("Density (log)")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    bad_weather_stats = (
        train_df.groupBy("bad_weather")
        .agg(
            F.count("*").alias("n"),
            F.avg("fare_amount").alias("mean_fare"),
            F.avg("hs_dist").alias("mean_dist"),
        )
        .orderBy("bad_weather")
    )

    try:
        display(bad_weather_stats)
    except NameError:
        bad_weather_stats.show(10, truncate=False)
else:
    print("bad_weather column not found; skipping bad weather impact section.")

### 6.3 Fare amount by weather/event category


In [0]:
# Average fare by event / weather category.

candidate_event_cols = ["event_cat", "weather_event", "event", "event_type"]
event_col = next((c for c in candidate_event_cols if c in train_df.columns), None)

if event_col is not None:
    event_stats = (
        train_df.groupBy(event_col)
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("fare_amount").alias("avg_fare_amount"),
            F.expr("percentile_approx(fare_amount, 0.5)").alias("median_fare_amount"),
        )
        .orderBy(F.col("trip_count").desc())
    )

    try:
        display(event_stats)
    except NameError:
        event_stats.show(50, truncate=False)

    # If the column is textual, pull out rain/snow-related categories (if present).
    if dict(train_df.dtypes).get(event_col, "").startswith("string"):
        rain_snow = event_stats.filter(
            F.lower(F.col(event_col)).contains("rain") | F.lower(F.col(event_col)).contains("snow")
        )
        try:
            display(rain_snow)
        except NameError:
            rain_snow.show(50, truncate=False)

else:
    flag_cols = [c for c in ["rain", "snow", "cold_weather"] if c in train_df.columns]
    if not flag_cols:
        print("No event category column found (e.g., 'event_cat') and no weather flags found; skipping event analysis.")
    else:
        for flag in flag_cols:
            stats = (
                train_df.groupBy(flag)
                .agg(
                    F.count("*").alias("trip_count"),
                    F.avg("fare_amount").alias("avg_fare_amount"),
                )
                .orderBy(flag)
            )
            print(f"\nFare stats grouped by '{flag}':")
            try:
                display(stats)
            except NameError:
                stats.show(10, truncate=False)


## 7. Modeling


### Linear Regression, XGBoost & LightGBM

#### Functions for ML models

In [0]:
TARGET = "fare_amount"
KEY_COL = "key"
SEED = 42

In [0]:
def rmse_np(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def r2_np(y_true, y_pred):
    return float(r2_score(y_true, y_pred))

In [0]:
def build_assembler(features, output_col="features"):
    return VectorAssembler(
        inputCols=features,
        outputCol=output_col,
        handleInvalid="skip"
    )

In [0]:
def prepare_xy_spark(df, features, target=TARGET, key_col=KEY_COL):
    cols = ([key_col] if key_col in df.columns else []) + features + ([target] if target in df.columns else [])
    sdf = df.select(*cols)
    assembler = build_assembler(features, output_col="features")

    if target in df.columns:
        out = assembler.transform(sdf).select(
            *( [key_col] if key_col in df.columns else [] ),
            "features",
            F.col(target).alias("label")
        )
    else:
        out = assembler.transform(sdf).select(
            *( [key_col] if key_col in df.columns else [] ),
            "features"
        )

    return out, assembler

In [0]:
def evaluate_spark(preds_df, label_col="label", pred_col="prediction"):
    evaluator_rmse = RegressionEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="rmse")
    evaluator_r2 = RegressionEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="r2")
    rmse_val = float(evaluator_rmse.evaluate(preds_df))
    r2_val = float(evaluator_r2.evaluate(preds_df))
    return rmse_val, r2_val

In [0]:
def train_lr_spark(train_df, features, target=TARGET):
    train_vec, assembler = prepare_xy_spark(train_df, features, target=target, key_col=KEY_COL)

    lr = SparkLinearRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=200,
        regParam=0.0,
        elasticNetParam=0.0
    )

    model = lr.fit(train_vec)
    preds = model.transform(train_vec)
    rmse_val, r2_val = evaluate_spark(preds)

    return model, assembler, rmse_val, r2_val

def predict_lr_spark(model, assembler, df, features, key_col=KEY_COL, pred_col="prediction"):
    vec = assembler.transform(df.select(key_col, *features)).select(key_col, "features")
    preds = model.transform(vec).select(key_col, F.col("prediction").alias(pred_col))
    return preds

In [0]:
def spark_to_pandas(df, features, target=None, key_col=KEY_COL, sample_frac=None, max_rows=None, seed=SEED):
    cols = ([key_col] if key_col in df.columns else []) + features + ([target] if target is not None else [])
    sdf = df.select(*cols)

    if target is not None:
        sdf = sdf.dropna(subset=features + [target])
    else:
        sdf = sdf.dropna(subset=features)

    if sample_frac is not None and sample_frac < 1.0:
        sdf = sdf.sample(withReplacement=False, fraction=sample_frac, seed=seed)

    if max_rows is not None:
        sdf = sdf.limit(int(max_rows))

    return sdf.toPandas()

In [0]:
def train_xgb_python(train_df_spark, features, target=TARGET, params=None, val_size=0.2, sample_frac=None, max_rows=None):
    if params is None:
        params = {}

    train_pd = spark_to_pandas(
        df=train_df_spark,
        features=features,
        target=target,
        key_col=KEY_COL,
        sample_frac=sample_frac,
        max_rows=max_rows
    )

    X = train_pd[features].to_numpy()
    y = train_pd[target].to_numpy()

    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=val_size, random_state=SEED)

    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr)

    val_pred = model.predict(X_val)
    rmse_val = rmse_np(y_val, val_pred)
    r2_val = r2_np(y_val, val_pred)

    return model, rmse_val, r2_val

def predict_xgb_python(test_df_spark, model, features, key_col=KEY_COL, sample_frac=None, max_rows=None):
    test_pd = spark_to_pandas(
        df=test_df_spark,
        features=features,
        target=None,
        key_col=key_col,
        sample_frac=sample_frac,
        max_rows=max_rows
    )

    preds = model.predict(test_pd[features].to_numpy())
    out = test_pd[[key_col]].copy()
    out["prediction"] = preds
    return out

In [0]:
def train_lgbm_python(train_df_spark, features, target=TARGET, params=None, val_size=0.2, sample_frac=None, max_rows=None):
    if params is None:
        params = {}

    train_pd = spark_to_pandas(
        df=train_df_spark,
        features=features,
        target=target,
        key_col=KEY_COL,
        sample_frac=sample_frac,
        max_rows=max_rows
    )

    X = train_pd[features].to_numpy()
    y = train_pd[target].to_numpy()

    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=val_size, random_state=SEED)

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    val_pred = model.predict(X_val, num_iteration=getattr(model, "best_iteration_", None))
    rmse_val = rmse_np(y_val, val_pred)
    r2_val = r2_np(y_val, val_pred)

    return model, rmse_val, r2_val

def predict_lgbm_python(test_df_spark, model, features, key_col=KEY_COL, sample_frac=None, max_rows=None):
    test_pd = spark_to_pandas(
        df=test_df_spark,
        features=features,
        target=None,
        key_col=key_col,
        sample_frac=sample_frac,
        max_rows=max_rows
    )

    preds = model.predict(test_pd[features].to_numpy(), num_iteration=getattr(model, "best_iteration_", None))
    out = test_pd[[key_col]].copy()
    out["prediction"] = preds
    return out

In [0]:
def preview_predictions_spark(preds_df, n=10):
    preds_df.show(n, truncate=False)

def preview_predictions_pandas(preds_pd, n=10):
    display(preds_pd.head(n))

def to_submission_df(preds, key_col=KEY_COL, pred_col="prediction", target_col=TARGET):
    if isinstance(preds, pd.DataFrame):
        out = preds.rename(columns={pred_col: target_col})
        return out[[key_col, target_col]]
    out = preds.select(key_col, F.col(pred_col).alias(target_col))
    return out

def save_submission(submission, filename):
    if isinstance(submission, pd.DataFrame):
        submission.to_csv(filename, index=False)
        return submission
    submission_pd = submission.toPandas()
    submission_pd.to_csv(filename, index=False)
    return submission_pd

In [0]:
BASE_FEATURES = [
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
    "passenger_count"
]

EDA_FEATURES = [
    "hs_dist",
    "tms_drop_distance",
    "plz_drop_distance",
    "met_drop_distance",
    "hbk_drop_distance",
    "pickup_longitude",
    "nyc_drop_distance",
    "wtc_drop_distance",
    "dropoff_longitude",
    "sol_drop_distance",
    "ewr_drop_distance",
    "lga_drop_distance",
    "passenger_count",
    "dropoff_latitude",
    "pickup_latitude",
    "jfk_drop_distance",
    "hour"
]

#### Linear Regression - baseline

In [0]:
lr_base_model, lr_base_assembler, lr_base_rmse, lr_base_r2 = train_lr_spark(train_df, BASE_FEATURES)
print("Linear Regression - Baseline")
print(f"RMSE: {lr_base_rmse:.4f}")
print(f"R2:   {lr_base_r2:.4f}")


In [0]:
lr_base_preds_spark = predict_lr_spark(lr_base_model, lr_base_assembler, test_df, EDA_FEATURES)
preview_predictions_spark(lr_base_preds_spark, n=5)

In [0]:
# lr_base_submission = to_submission_df(lr_base_preds_spark)
# save_submission(lr_base_submission, "lr_base_submission.csv")

#### Linear Regression - with features chosen after EDA

In [0]:
lr_model, lr_assembler, lr_rmse, lr_r2 = train_lr_spark(train_df, EDA_FEATURES)
print("Linear Regression - EDA features")
print(f"RMSE: {lr_rmse:.4f}")
print(f"R2:   {lr_r2:.4f}")

In [0]:
lr_preds_spark = predict_lr_spark(lr_model, lr_assembler, test_df, EDA_FEATURES)
preview_predictions_spark(lr_preds_spark, n=5)

In [0]:
# lr_submission = to_submission_df(lr_preds_spark)
# save_submission(lr_submission, "lr_submission.csv")

#### XGBoost

In [0]:
xgb_params = {
    "n_estimators": 500,
    "max_depth": 8,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "n_jobs": 4,
    "random_state": SEED,
    "tree_method": "hist",
}

In [0]:
xgb_model, xgb_rmse, xgb_r2 = train_xgb_python(
    train_df_spark=train_df,
    features=EDA_FEATURES,
    target=TARGET,
    params=xgb_params,
    val_size=0.2,
    sample_frac=None,
    max_rows=None
)
print("XGBoost")
print(f"RMSE: {xgb_rmse:.4f}")
print(f"R2:   {xgb_r2:.4f}")

In [0]:
xgb_preds_pd = predict_xgb_python(test_df, xgb_model, EDA_FEATURES)
preview_predictions_pandas(xgb_preds_pd, n=5)

In [0]:
# xgb_submission = to_submission_df(xgb_preds_pd)
# save_submission(xgb_submission, "xgb_submission.csv")

#### LGBM

In [0]:
lgb_params = dict(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=4
)

In [0]:
lgb_model, lgb_rmse, lgb_r2 = train_lgbm_python(
    train_df_spark=train_df,
    features=EDA_FEATURES,
    target=TARGET,
    params=lgb_params,
    val_size=0.2,
    sample_frac=None,
    max_rows=None
)

print("LightGBM")
print(f"RMSE: {lgb_rmse:.4f}")
print(f"R2:   {lgb_r2:.4f}")

In [0]:
lgb_preds_pd = predict_lgbm_python(test_df, lgb_model, EDA_FEATURES)
preview_predictions_pandas(lgb_preds_pd, n=5)

In [0]:
# lgb_submission = to_submission_df(lgb_preds_pd)
# save_submission(lgb_submission, "lgb_submission.csv")

In [0]:
MODEL_PATH = "lgb_model.txt"

def save_lgbm_model(lgb_model, path=MODEL_PATH):
    booster = lgb_model.booster_
    booster.save_model(path)
    return path

In [0]:
save_lgbm_model(lgb_model, "lgb_model.txt")

## 8. Prediction / Scoring


### Predicting fare amount

In [0]:
def predict_fare_from_values_booster(booster, features, values_dict):
    row = {f: values_dict[f] for f in features}
    X = pd.DataFrame([row], columns=features)
    pred = float(booster.predict(X)[0])
    return pred

In [0]:
def load_lgbm_model(path=MODEL_PATH):
    return lgb.Booster(model_file=path)

In [0]:
FEATURES = EDA_FEATURES
FEATURES

In [0]:
values = {
    # computed feature
    "hs_dist": 4.25,

    # distances from dropoff to landmarks (km or miles – same unit as training)
    "tms_drop_distance": 2.1,
    "plz_drop_distance": 1.8,
    "met_drop_distance": 3.4,
    "hbk_drop_distance": 4.0,
    "nyc_drop_distance": 0.9,
    "wtc_drop_distance": 1.2,
    "sol_drop_distance": 2.6,
    "ewr_drop_distance": 14.8,
    "lga_drop_distance": 9.3,
    "jfk_drop_distance": 21.7,

    # raw coordinates
    "pickup_longitude": -74.02,
    "pickup_latitude": 40.75,
    "dropoff_longitude": -73.98,
    "dropoff_latitude": 40.77,

    # other features
    "passenger_count": 1,
    "hour": 5,
}

In [0]:
booster = load_lgbm_model("lgb_model.txt")
fare = predict_fare_from_values_booster(booster, EDA_FEATURES, values)
print(f"Predicted fare_amount: {fare:.2f}")